# 🗄️ Consultas directas a la Base de Datos EcoMarket

Notebook para explorar y consultar la BD sin pasar por el chatbot.

**Requisitos:**
- Haber ejecutado `01_base_datos.ipynb` para generar `data/ecomarket.db`
- Ambiente `ia_muba` activado

In [52]:
import sqlite3
import os
from datetime import datetime
import pandas as pd

DB_PATH = '../data/ecomarket.db'

# Verificar que la BD existe
if os.path.exists(DB_PATH):
    print(f'✅ Base de datos encontrada: {DB_PATH}')
else:
    print(f'❌ No se encontró la BD. Ejecuta primero 01_base_datos.ipynb')

def get_connection():
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    return conn

✅ Base de datos encontrada: ../data/ecomarket.db


## 1. Explorar estructura de la BD

In [53]:
# Ver todas las tablas
conn = sqlite3.connect(DB_PATH)
tablas = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", conn
)
print('📋 Tablas en la base de datos:')
tablas

📋 Tablas en la base de datos:


,name
0,detalle_pedido
1,devoluciones
2,pedidos
3,productos
4,promociones
5,sqlite_sequence
6,tickets_soporte


In [54]:
# Conteo de registros por tabla
for tabla in tablas['name']:
    count = pd.read_sql_query(f'SELECT COUNT(*) as total FROM {tabla}', conn).iloc[0]['total']
    print(f'  {tabla:<25} → {count} registros')

  detalle_pedido            → 227 registros
  devoluciones              → 8 registros
  pedidos                   → 54 registros
  productos                 → 72 registros
  promociones               → 13 registros
  sqlite_sequence           → 6 registros
  tickets_soporte           → 9 registros


In [55]:
# Ver esquema de una tabla (cambiar el nombre según necesites)
TABLA = 'productos'  # 👈 Cambia aquí: productos, pedidos, detalle_pedido, promociones, devoluciones, tickets_soporte

esquema = pd.read_sql_query(f"PRAGMA table_info({TABLA})", conn)
print(f'🏗️ Esquema de "{TABLA}":')
esquema[['name', 'type', 'notnull', 'pk']]

🏗️ Esquema de "productos":


,name,type,notnull,pk
0,id,INTEGER,0,1
1,nombre,TEXT,1,0
2,categoria,TEXT,1,0
3,precio,REAL,1,0
4,stock,INTEGER,1,0
5,descripcion,TEXT,0,0
6,unidad,TEXT,0,0


## 2. Consultar Productos

In [56]:
# Todos los productos
df_productos = pd.read_sql_query(
    'SELECT id, nombre, categoria, precio, stock, unidad FROM productos ORDER BY categoria, nombre',
    conn
)
print(f'🛒 Total productos: {len(df_productos)}')
df_productos

🛒 Total productos: 72


,id,nombre,categoria,precio,stock,unidad
0,22,Agua Mineral,Bebidas,0.7,500,botella
1,26,Cafe Molido,Bebidas,5.0,59,unidad
2,25,Cerveza Artesanal,Bebidas,3.5,80,botella
3,24,Coca-Cola,Bebidas,1.8,200,botella
4,27,Te Verde,Bebidas,3.2,70,caja
...,...,...,...,...,...,...
67,65,Papa,Verduras,2.0,150,kg
68,72,Pepino,Verduras,1.8,100,unidad
69,71,Pimentón,Verduras,2.8,90,unidad
70,3,Tomate,Verduras,3.0,100,kg


In [57]:
# Buscar producto por nombre
BUSCAR = 'pan'  # 👈 Cambia aquí

pd.read_sql_query(
    'SELECT id, nombre, categoria, precio, stock, unidad FROM productos WHERE LOWER(nombre) LIKE LOWER(?)',
    conn, params=(f'%{BUSCAR}%',)
)

,id,nombre,categoria,precio,stock,unidad
0,18,Pan Integral,Panaderia,2.1,99,unidad
1,44,Pescado Empanado,Congelados,5.5,55,paquete
2,58,Panales Talla 3,Infantil,14.0,40,paquete
3,62,Pan de Pita,Panaderia,2.8,74,paquete


In [58]:
# Productos por categoría
CATEGORIA = 'frutas'  # 👈 Cambia aquí

pd.read_sql_query(
    'SELECT id, nombre, precio, stock, unidad FROM productos WHERE LOWER(categoria) LIKE LOWER(?) ORDER BY nombre',
    conn, params=(f'%{CATEGORIA}%',)
)

,id,nombre,precio,stock,unidad
0,5,Aguacate,4.5,60,unidad
1,69,Mandarina,2.0,100,kg
2,67,Mango,3.5,77,unidad
3,1,Manzana Roja,2.5,148,kg
4,7,Naranja,2.0,179,kg
5,68,Pera,2.2,90,kg
6,2,Platano,1.8,198,kg


In [59]:
# Categorías disponibles
pd.read_sql_query('SELECT DISTINCT categoria FROM productos ORDER BY categoria', conn)

,categoria
0,Bebidas
1,Carnes
2,Congelados
3,Cuidado Personal
4,Despensa
5,Frutas
6,Infantil
7,Lacteos
8,Limpieza y Hogar
9,Mascotas


## 3. Consultar Pedidos

In [60]:
# Todos los pedidos
df_pedidos = pd.read_sql_query(
    'SELECT id, email_cliente, fecha_pedido, estado, total, direccion_envio, fecha_entrega_estimada FROM pedidos ORDER BY fecha_pedido DESC',
    conn
)
print(f'📦 Total pedidos: {len(df_pedidos)}')
df_pedidos

📦 Total pedidos: 54


,id,email_cliente,fecha_pedido,estado,total,direccion_envio,fecha_entrega_estimada
0,25,diego.moreno@email.com,2026-06-19,enviado,117.05,"Gran Via 45, Valencia",2026-06-24
1,36,pedro.sanchez@email.com,2026-06-19,confirmado,89.32,"Calle Mayor 15, Madrid",2026-06-21
2,51,Danna.sarmiento@email.com,2026-06-19,confirmado,31.20,calle salitre 13,2026-06-21
3,52,test@test.com,2026-06-19,confirmado,4.50,"Calle Gran Vía 10, Madrid",2026-06-21
4,53,test@test.com,2026-06-19,confirmado,3.80,"Av. de Andalucía 5, Málaga",2026-06-21
5,54,Roberto.romero@email.com,2026-06-19,confirmado,18.30,"Calle Salitre 13, Málaga",2026-06-21
6,16,antonio.jimenez@email.com,2026-06-18,enviado,57.12,"Av. Diagonal 200, Barcelona",2026-06-21
7,7,carlos.lopez@email.com,2026-06-17,entregado,25.16,"Calle Alcala 78, Madrid",2026-06-24
8,9,sofia.ruiz@email.com,2026-06-17,cancelado,45.78,"Av. de la Constitucion 12, Bilbao",2026-06-22
9,30,carlos.lopez@email.com,2026-06-17,entregado,100.68,"Calle Real 33, Zaragoza",2026-06-21


In [61]:
# Pedidos de un cliente específico
EMAIL = 'ana.martinez@email.com'  # 👈 Cambia aquí

pd.read_sql_query(
    'SELECT id, fecha_pedido, estado, total, direccion_envio FROM pedidos WHERE LOWER(email_cliente) = LOWER(?) ORDER BY fecha_pedido DESC',
    conn, params=(EMAIL,)
)

,id,fecha_pedido,estado,total,direccion_envio
0,31,2026-05-29,entregado,114.42,"Gran Via 45, Valencia"
1,2,2026-05-27,confirmado,86.05,"Calle Real 33, Zaragoza"


In [62]:
# Detalle de un pedido específico
PEDIDO_ID = 1  # 👈 Cambia aquí

pd.read_sql_query('''
    SELECT p.nombre, dp.cantidad, dp.precio_unitario,
           (dp.cantidad * dp.precio_unitario) as subtotal
    FROM detalle_pedido dp
    JOIN productos p ON p.id = dp.producto_id
    WHERE dp.pedido_id = ?
''', conn, params=(PEDIDO_ID,))

,nombre,cantidad,precio_unitario,subtotal
0,Pan Integral,4,2.1,8.4
1,Zanahoria,3,1.2,3.6


## 4. Consultar Promociones

In [63]:
# Promociones vigentes
hoy = datetime.now().strftime('%Y-%m-%d')
print(f'📅 Fecha actual: {hoy}')

pd.read_sql_query('''
    SELECT pr.id, p.nombre as producto, pr.descripcion,
           pr.descuento_porcentaje as descuento_pct,
           p.precio as precio_original,
           ROUND(p.precio * (1 - pr.descuento_porcentaje/100.0), 2) as precio_final,
           pr.fecha_inicio, pr.fecha_fin
    FROM promociones pr
    JOIN productos p ON p.id = pr.producto_id
    WHERE pr.activa = 1 AND pr.fecha_inicio <= ? AND pr.fecha_fin >= ?
    ORDER BY pr.descuento_porcentaje DESC
''', conn, params=(hoy, hoy))

📅 Fecha actual: 2026-06-19


,id,producto,descripcion,descuento_pct,precio_original,precio_final,fecha_inicio,fecha_fin
0,1,Manzana Roja,2x1 en Manzanas Rojas,50.0,2.5,1.25,2026-06-16,2026-06-26
1,6,Pan Integral,Pan Integral a mitad de precio los martes,50.0,2.1,1.05,2026-06-19,2026-07-03
2,9,Pizza Margarita,Pizza Margarita: 2da unidad al 50%,50.0,4.2,2.10,2026-06-19,2026-06-26
3,4,Agua Mineral,Agua Mineral: 3x2,33.0,0.7,0.47,2026-06-16,2026-07-03
4,10,Patatas Fritas,Patatas Fritas: 3x2,33.0,1.9,1.27,2026-06-16,2026-07-03
5,7,Chocolate Negro,25% descuento en Chocolate Negro,25.0,3.5,2.63,2026-06-16,2026-06-26
6,8,Cerveza Artesanal,Cerveza Artesanal: 4x3,25.0,3.5,2.63,2026-06-19,2026-07-03
7,2,Leche Entera,20% descuento en Leche Entera,20.0,1.2,0.96,2026-06-19,2026-07-03
8,11,Champu Suave,Champu Suave: 20% descuento,20.0,3.5,2.80,2026-06-19,2026-07-03
9,3,Pechuga de Pollo,15% descuento en Pechuga de Pollo,15.0,6.9,5.87,2026-06-19,2026-06-26


## 5. Consultar Devoluciones

In [64]:
# Todas las devoluciones
pd.read_sql_query('''
    SELECT d.id, d.pedido_id, d.email_cliente, d.motivo, d.estado,
           d.fecha_solicitud, d.fecha_resolucion
    FROM devoluciones d
    ORDER BY d.fecha_solicitud DESC
''', conn)

,id,pedido_id,email_cliente,motivo,estado,fecha_solicitud,fecha_resolucion
0,2,7,carlos.lopez@email.com,Producto equivocado,pendiente,2026-06-17,None
1,5,25,elena.torres@email.com,Envase roto en la entrega,pendiente,2026-06-17,None
2,3,12,laura.fernandez@email.com,No corresponde a lo solicitado,en_revision,2026-06-16,None
3,6,31,miguel.herrera@email.com,Calidad inferior a la esperada,en_revision,2026-06-16,None
4,1,3,ana.martinez@email.com,Producto llego danado,aprobada,2026-06-14,2026-06-17
5,4,18,pedro.sanchez@email.com,Producto caducado al recibir,aprobada,2026-06-14,2026-06-17
6,7,38,carmen.vargas@email.com,Pedido incompleto,rechazada,2026-06-14,2026-06-17
7,8,44,pablo.navarro@email.com,Producto no fresco,aprobada,2026-06-14,2026-06-17


## 6. Tickets de Soporte

In [65]:
pd.read_sql_query('''
    SELECT id, email_cliente, asunto, estado, fecha_creacion, pedido_id
    FROM tickets_soporte
    ORDER BY fecha_creacion DESC
''', conn)

,id,email_cliente,asunto,estado,fecha_creacion,pedido_id
0,9,Roberto.romero@email.com,Solicitud de cancelación de pedido,abierto,2026-06-19 11:22:50,NaN
1,6,elena.torres@email.com,Factura no recibida,abierto,2026-06-19 09:50:19,25.0
2,4,juan.rodriguez@email.com,Cambio de direccion,abierto,2026-06-18 09:50:19,22.0
3,7,miguel.herrera@email.com,Problema con promocion,abierto,2026-06-18 09:50:19,28.0
4,1,maria.garcia@email.com,Pedido no recibido,abierto,2026-06-15 09:50:19,5.0
5,5,sofia.ruiz@email.com,Producto agotado,en_proceso,2026-06-15 09:50:19,15.0
6,2,carlos.lopez@email.com,Cobro duplicado,en_proceso,2026-06-12 09:50:19,7.0
7,3,ana.martinez@email.com,Consulta alergenos,cerrado,2026-06-09 09:50:19,NaN
8,8,isabel.romero@email.com,Queja por demora,cerrado,2026-06-09 09:50:19,33.0


## 7. Consulta SQL Libre

Escribe cualquier consulta SELECT que necesites:

In [66]:
# 👇 Escribe tu consulta SQL aquí
MI_CONSULTA = '''
SELECT * FROM productos WHERE stock < 50
'''

pd.read_sql_query(MI_CONSULTA, conn)

,id,nombre,categoria,precio,stock,descripcion,unidad
0,10,Queso Manchego,Lacteos,8.5,40,Queso manchego curado 250g,unidad
1,14,Carne Molida,Carnes,8.5,40,"Carne molida de res, 500g",unidad
2,15,Salmon Fresco,Carnes,12.0,30,"Filete de salmon fresco, 300g",unidad
3,16,Jamon Serrano,Carnes,15.0,25,Jamon serrano loncheado 200g,unidad
4,40,Miel,Despensa,5.5,39,Miel pura de flores 500g,unidad
5,55,Pienso Perro Adulto,Mascotas,12.5,45,Pienso seco para perro adulto 3kg,saco
6,58,Panales Talla 3,Infantil,14.0,40,Panales talla 3 (4-9 kg) x44,paquete
7,59,Leche Infantil,Infantil,11.5,35,Leche de continuacion 800g,lata
8,64,Carne de Res,Carnes,12.0,40,"Carne de res fresca para asar, 1kg",kg


In [67]:
# Cerrar conexión al terminar
conn.close()
print('🔒 Conexión cerrada.')

🔒 Conexión cerrada.
